# 01 - Audit des données (Phase 2)
**Objectif :** comprendre le dataset *Online Retail II* et **quantifier** ses défauts avant tout nettoyage.
Tous les chiffres de ce notebook alimentent la section « Compréhension des données » et « Limites » du rapport.

> Prérequis : placer `online_retail_II.xlsx` dans `data/raw/` (ou changer `paths.raw_file` dans `config.yaml`).
> La première lecture du xlsx prend 1 à 3 minutes ; un cache parquet est ensuite utilisé.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, resolve, save_table, load_table
cfg = load_config()
FIG = resolve(cfg, "figures_dir")
OUT = resolve(cfg, "outputs_dir")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .3})

from src.cleaning import load_raw, audit_summary, clean, quality_bias_table
RAW = ROOT / cfg["paths"]["raw_file"]
CACHE = resolve(cfg, "interim_dir") / "raw_merged.parquet"
print("Fichier brut :", RAW)

## 1. Chargement et fusion des deux feuilles

In [ ]:
df = load_raw(RAW, CACHE)
print(f"{len(df):,} lignes x {df.shape[1]} colonnes")
print(df.groupby("SourceSheet")["InvoiceDate"].agg(["count", "min", "max"]))
df.head()

In [ ]:
df.dtypes

## 2. Chevauchement entre les deux feuilles
Les deux feuilles partagent les premiers jours de décembre 2010. Les lignes présentes dans **les deux** sont des doublons exacts à supprimer après fusion.

In [ ]:
body = df.drop(columns=["SourceSheet"])
overlap = df[body.duplicated(keep=False)]
print("Lignes impliquées dans un doublon exact :", len(overlap))
print("Doublons à supprimer (garder 1 exemplaire) :", int(body.duplicated().sum()))
if len(overlap):
    print("Période concernée :", overlap["InvoiceDate"].min(), "->", overlap["InvoiceDate"].max())
    print(overlap.groupby("SourceSheet").size())

## 3. Tableau de synthèse qualité (avant nettoyage)

In [ ]:
summary = audit_summary(df, cfg)
summary.to_csv(OUT / "audit_summary.csv", index=False)
summary

## 4. Valeurs manquantes

In [ ]:
miss = df.isna().agg(["sum", "mean"]).T
miss["mean"] = (100 * miss["mean"]).round(2)
miss.columns = ["manquantes", "% manquantes"]
miss

## 5. Clients absents (`CustomerID`) : poids et biais
Les lignes sans identifiant ne peuvent entrer ni dans le RFM, ni dans le churn, ni dans la recommandation par client.
Il faut mesurer **combien elles pèsent** et **en quoi elles diffèrent**.

In [ ]:
rev = df["Quantity"] * df["Price"]
no_id = df["CustomerID"].isna()
print(f"% lignes sans ID : {100*no_id.mean():.1f} %")
print(f"% du CA brut sans ID : {100*rev[no_id].sum()/rev.sum():.1f} %")

## 6. Annulations, quantités et prix anormaux

In [ ]:
prefix = df["Invoice"].str.extract(r"^([A-Za-z]+)")[0].fillna("(numérique)")
print("Préfixes de facture :"); print(prefix.value_counts())
print("\nQuantity <= 0 :", int((df["Quantity"] <= 0).sum()), "| Price <= 0 :", int((df["Price"] <= 0).sum()))
neg_no_c = (df["Quantity"] < 0) & ~df["Invoice"].str.startswith("C")
print("Quantity < 0 SANS préfixe C (ajustements de stock) :", int(neg_no_c.sum()))
print("\nQuantité : percentiles"); print(df["Quantity"].quantile([0, .001, .01, .5, .99, .999, 1]))
print("\nPrix : percentiles"); print(df["Price"].quantile([0, .001, .01, .5, .99, .999, 1]))

## 7. Codes produits non-produits et descriptions incohérentes

In [ ]:
non_alpha = df.loc[~df["StockCode"].str.match(r"^\d{5}[A-Z]*$"), "StockCode"].value_counts()
print("StockCode qui ne ressemblent pas à un code produit (5 chiffres + lettres) :")
print(non_alpha.head(25))
multi = df.groupby("StockCode")["Description"].nunique()
print("\nStockCode avec plusieurs descriptions :", int((multi > 1).sum()), "sur", len(multi))
print(df[df["StockCode"].isin(multi[multi > 1].index[:3])].groupby(["StockCode", "Description"]).size())

## 8. Pays

In [ ]:
c = df["Country"].value_counts()
print(c.head(10)); print(f"\nRoyaume-Uni : {100*c.get('United Kingdom', 0)/c.sum():.1f} % des lignes")

## 9. Nettoyage selon la politique documentée (`src/cleaning.py`)
Chaque règle est appliquée dans l'ordre et tracée dans le tableau **« lignes exclues et pourquoi »**, à copier dans le rapport.

In [ ]:
clean_df, log = clean(df, cfg)
log.to_csv(OUT / "exclusion_log.csv", index=False)
log

In [ ]:
print("Biais : factures avec vs sans CustomerID")
quality_bias_table(clean_df)

## 10. À retenir pour le rapport
Compléter après exécution :
- part de lignes et de CA sans `CustomerID` : ...
- nombre de doublons (dont chevauchement) : ...
- retours (facture `C`) : ...% des lignes
- lignes exclues au total : ...%
- les factures sans ID diffèrent-elles (panier, pays) ? -> limite à signaler